# 🚀 Algorithmic Trading System - Exploratory Analysis

This notebook demonstrates the core components of our algorithmic trading and portfolio optimization system.

## 📋 Overview

We'll walk through:
1. **Data Loading & Exploration**
2. **Feature Engineering & Technical Indicators**
3. **Time Series Forecasting (ARIMA + GARCH)**
4. **Signal Generation**
5. **Portfolio Optimization**
6. **Backtesting & Performance Evaluation**

Let's start by importing the necessary libraries and setting up our environment.

In [ ]:
# Import standard libraries
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime

# Configure plotting
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)
warnings.filterwarnings('ignore')

# Add src directory to path
sys.path.append('../src')

# Import our custom modules
from data_loader import load_data
from feature_engineering import make_features
from forecasting import ARIMAGARCHForecaster
from signal_generator import SignalGenerator
from optimizer import PortfolioOptimizer
from backtester import Backtester
from evaluator import PerformanceEvaluator
from utils import TradingConfig, setup_logging

print("✅ All modules imported successfully!")

## 1. 📊 Data Loading & Exploration

Let's start by loading historical price data for a few popular assets and exploring the data structure.

In [ ]:
# Define our assets and time period
tickers = ['AAPL', 'MSFT', 'GOOGL', 'SPY']  # Tech stocks + market index
start_date = '2020-01-01'
end_date = '2024-01-01'

print(f"📈 Loading data for {tickers}")
print(f"📅 Period: {start_date} to {end_date}")

# Load the data
full_data, price_data = load_data(
    tickers=tickers,
    start=start_date,
    end=end_date
)

print(f"\n✅ Data loaded successfully!")
print(f"📊 Shape: {price_data.shape[0]} days, {price_data.shape[1]} assets")
print(f"📍 Date range: {price_data.index[0].date()} to {price_data.index[-1].date()}")

In [ ]:
# Display first few rows
print("📋 First 5 rows of price data:")
price_data.head()

In [ ]:
# Basic statistics
print("📊 Descriptive Statistics:")
price_data.describe()

In [ ]:
# Visualize price evolution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for i, ticker in enumerate(tickers):
    # Normalize to show relative performance
    normalized_price = price_data[ticker] / price_data[ticker].iloc[0]
    
    axes[i].plot(normalized_price.index, normalized_price.values, linewidth=2, label=ticker)
    axes[i].set_title(f'{ticker} - Normalized Price Evolution', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Normalized Price')
    axes[i].grid(True, alpha=0.3)
    axes[i].legend()

plt.tight_layout()
plt.suptitle('📈 Asset Price Evolution (Normalized to Starting Value)', fontsize=16, y=1.02)
plt.show()

# All assets on one plot for comparison
plt.figure(figsize=(14, 8))
for ticker in tickers:
    normalized_price = price_data[ticker] / price_data[ticker].iloc[0]
    plt.plot(normalized_price.index, normalized_price.values, linewidth=2, label=ticker, alpha=0.8)

plt.title('📊 Relative Performance Comparison', fontsize=16, fontweight='bold')
plt.ylabel('Normalized Price')
plt.xlabel('Date')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. 🔧 Feature Engineering & Technical Indicators

Now let's compute various technical indicators and features that will be used for signal generation.

In [ ]:
# Generate comprehensive features
print("🔧 Computing technical indicators and features...")

features = make_features(
    prices=price_data,
    include_returns=True,
    include_ma=True,
    include_volatility=True,
    include_rsi=True,
    include_macd=True,
    include_bb=True,
    include_stats=True,
    include_momentum=True
)

print(f"✅ Generated {len(features)} feature sets:")
for name, data in features.items():
    if hasattr(data, 'shape'):
        print(f"  📊 {name}: {data.shape}")
    else:
        print(f"  📊 {name}: {type(data)}")

In [ ]:
# Visualize daily returns
returns = features['returns']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for i, ticker in enumerate(tickers):
    axes[i].plot(returns.index, returns[ticker], alpha=0.7, linewidth=0.8)
    axes[i].set_title(f'{ticker} - Daily Returns', fontweight='bold')
    axes[i].set_ylabel('Return')
    axes[i].grid(True, alpha=0.3)
    
    # Add mean line
    mean_return = returns[ticker].mean()
    axes[i].axhline(y=mean_return, color='red', linestyle='--', alpha=0.8, 
                   label=f'Mean: {mean_return:.4f}')
    axes[i].legend()

plt.tight_layout()
plt.suptitle('📊 Daily Returns Distribution', fontsize=16, y=1.02)
plt.show()

In [ ]:
# RSI visualization
rsi = features['RSI']

plt.figure(figsize=(14, 10))

for i, ticker in enumerate(tickers):
    plt.subplot(2, 2, i+1)
    plt.plot(rsi.index, rsi[ticker], linewidth=1.5, label=f'{ticker} RSI')
    
    # Add overbought/oversold lines
    plt.axhline(y=70, color='red', linestyle='--', alpha=0.7, label='Overbought (70)')
    plt.axhline(y=30, color='green', linestyle='--', alpha=0.7, label='Oversold (30)')
    plt.axhline(y=50, color='gray', linestyle='-', alpha=0.5)
    
    plt.title(f'{ticker} - RSI (14-day)', fontweight='bold')
    plt.ylabel('RSI')
    plt.ylim(0, 100)
    plt.grid(True, alpha=0.3)
    plt.legend()

plt.tight_layout()
plt.suptitle('📈 Relative Strength Index (RSI)', fontsize=16, y=1.02)
plt.show()

In [ ]:
# Correlation matrix
plt.figure(figsize=(10, 8))
correlation_matrix = returns.corr()

sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
            square=True, fmt='.3f', cbar_kws={'shrink': 0.8})
plt.title('🔗 Asset Return Correlations', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("📊 Correlation Analysis:")
print(correlation_matrix)

## 3. 🔮 Time Series Forecasting (ARIMA + GARCH)

Let's demonstrate the forecasting capabilities using our combined ARIMA + GARCH model.

In [ ]:
# Initialize forecaster
print("🔮 Setting up ARIMA-GARCH forecasting...")

forecaster = ARIMAGARCHForecaster(
    arima_order=(1, 0, 1),
    garch_order=(1, 1),
    auto_order=True  # Automatically select optimal orders
)

# Generate forecasts
print("📊 Generating 5-day ahead forecasts...")
mean_forecasts, vol_forecasts = forecaster.forecast_portfolio(
    returns=returns,
    steps=5
)

print(f"✅ Forecasts generated for {len(mean_forecasts.columns)} assets")
print(f"📅 Forecast horizon: {len(mean_forecasts)} days")

In [ ]:
# Display forecast results
print("📈 Mean Return Forecasts (next 5 days):")
print(mean_forecasts)

print("\n📊 Volatility Forecasts (next 5 days):")
print(vol_forecasts)

In [ ]:
# Visualize forecasts
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Mean forecasts
mean_forecasts.T.plot(kind='bar', ax=axes[0], alpha=0.8)
axes[0].set_title('📈 Expected Return Forecasts (5-day ahead)', fontweight='bold')
axes[0].set_ylabel('Expected Return')
axes[0].grid(True, alpha=0.3)
axes[0].legend(title='Days Ahead')

# Volatility forecasts
vol_forecasts.T.plot(kind='bar', ax=axes[1], alpha=0.8, color='orange')
axes[1].set_title('📊 Volatility Forecasts (5-day ahead)', fontweight='bold')
axes[1].set_ylabel('Expected Volatility')
axes[1].grid(True, alpha=0.3)
axes[1].legend(title='Days Ahead')

plt.tight_layout()
plt.show()

## 4. 📡 Signal Generation

Now let's generate trading signals using multiple strategies and combine them.

In [ ]:
# Initialize signal generator
print("📡 Setting up signal generation...")

signal_generator = SignalGenerator(
    signal_threshold=0.1,
    volatility_scaling=True,
    signal_smoothing=True,
    smoothing_window=3
)

# Prepare data for signal generation
signal_data = {
    'prices': price_data,
    'returns': returns,
    'volatility': features['volatility'],
    'mean_forecast': mean_forecasts.iloc[[0]],  # Use first forecast
    'vol_forecast': vol_forecasts.iloc[[0]]
}

print("🎯 Generating individual strategy signals...")

In [ ]:
# Generate individual signals
ma_signals = signal_generator.momentum_ma_crossover(price_data, fast_window=5, slow_window=20)
rsi_signals = signal_generator.mean_reversion_rsi(price_data, oversold_threshold=30, overbought_threshold=70)
macd_signals = signal_generator.momentum_macd(price_data)

print(f"📊 Signal Statistics:")
print(f"  MA Crossover signals: {(ma_signals != 0).sum().sum()}")
print(f"  RSI signals: {(rsi_signals != 0).sum().sum()}")
print(f"  MACD signals: {(macd_signals != 0).sum().sum()}")

In [ ]:
# Generate combined signals
strategies = ['ma_crossover', 'macd', 'rsi', 'forecast']
strategy_weights = {
    'ma_crossover': 0.3,
    'macd': 0.2,
    'rsi': 0.2,
    'forecast': 0.3
}

combined_signals = signal_generator.generate_signals(
    data=signal_data,
    strategies=strategies,
    strategy_weights=strategy_weights
)

print(f"🎯 Combined signals generated: {(combined_signals != 0).sum().sum()} non-zero signals")

In [ ]:
# Visualize signals for one asset (AAPL)
asset = 'AAPL'
recent_period = slice(-252, None)  # Last year

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Price and MA signals
axes[0].plot(price_data.index[recent_period], price_data[asset][recent_period], 
            label=f'{asset} Price', linewidth=2)
axes[0].plot(price_data.index[recent_period], 
            price_data[asset][recent_period].rolling(5).mean(), 
            label='MA-5', alpha=0.8)
axes[0].plot(price_data.index[recent_period], 
            price_data[asset][recent_period].rolling(20).mean(), 
            label='MA-20', alpha=0.8)

# Add signal markers
ma_buy_signals = ma_signals[asset][recent_period] == 1
ma_sell_signals = ma_signals[asset][recent_period] == -1

if ma_buy_signals.any():
    axes[0].scatter(ma_buy_signals[ma_buy_signals].index, 
                   price_data[asset][ma_buy_signals[ma_buy_signals].index],
                   color='green', marker='^', s=100, label='MA Buy')
if ma_sell_signals.any():
    axes[0].scatter(ma_sell_signals[ma_sell_signals].index,
                   price_data[asset][ma_sell_signals[ma_sell_signals].index],
                   color='red', marker='v', s=100, label='MA Sell')

axes[0].set_title(f'{asset} - Price and Moving Average Signals', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# RSI and signals
axes[1].plot(rsi.index[recent_period], rsi[asset][recent_period], 
            label=f'{asset} RSI', linewidth=2)
axes[1].axhline(y=70, color='red', linestyle='--', alpha=0.7)
axes[1].axhline(y=30, color='green', linestyle='--', alpha=0.7)
axes[1].set_title(f'{asset} - RSI and Mean Reversion Signals', fontweight='bold')
axes[1].set_ylabel('RSI')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Combined signals
axes[2].plot(combined_signals.index[recent_period], combined_signals[asset][recent_period], 
            linewidth=2, label='Combined Signal')
axes[2].axhline(y=0, color='black', linestyle='-', alpha=0.5)
axes[2].set_title(f'{asset} - Combined Trading Signals', fontweight='bold')
axes[2].set_ylabel('Signal Strength')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 📈 Portfolio Optimization

Let's optimize our portfolio weights using the forecasted returns and covariance matrix.

In [ ]:
# Initialize portfolio optimizer
print("📈 Setting up portfolio optimization...")

optimizer = PortfolioOptimizer(
    risk_free_rate=0.02,
    max_weight=0.4,
    min_weight=0.0,
    transaction_cost=0.001
)

# Get inputs for optimization
next_returns = mean_forecasts.iloc[0]  # Next period forecast
cov_matrix = features['cov']

print(f"🎯 Optimizing portfolio for {len(next_returns)} assets")
print(f"📊 Expected returns: {next_returns.values}")

In [ ]:
# Test different optimization methods
methods = ['sharpe', 'mean_variance', 'risk_parity']
optimization_results = {}

for method in methods:
    try:
        if method == 'mean_variance':
            weights = optimizer.optimize_portfolio_forecasted(
                next_returns, cov_matrix, method, risk_aversion=2.0)
        else:
            weights = optimizer.optimize_portfolio_forecasted(
                next_returns, cov_matrix, method)
        
        optimization_results[method] = {
            'weights': weights,
            'expected_return': optimizer.last_returns,
            'volatility': optimizer.last_volatility,
            'sharpe': optimizer.last_sharpe
        }
        
        print(f"✅ {method.upper()} optimization completed")
        
    except Exception as e:
        print(f"❌ {method} optimization failed: {e}")

print(f"\n📊 Optimization completed for {len(optimization_results)} methods")

In [ ]:
# Display optimization results
results_df = pd.DataFrame(index=tickers)

for method, result in optimization_results.items():
    results_df[method] = result['weights']

print("📊 Optimal Portfolio Weights:")
print(results_df.round(4))

# Display performance metrics
print("\n📈 Expected Performance Metrics:")
metrics_df = pd.DataFrame({
    'Expected Return': [res['expected_return'] for res in optimization_results.values()],
    'Volatility': [res['volatility'] for res in optimization_results.values()],
    'Sharpe Ratio': [res['sharpe'] for res in optimization_results.values()]
}, index=optimization_results.keys())

print(metrics_df.round(4))

In [ ]:
# Visualize portfolio weights
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (method, result) in enumerate(optimization_results.items()):
    weights = result['weights']
    
    axes[i].pie(weights, labels=tickers, autopct='%1.1f%%', startangle=90)
    axes[i].set_title(f'{method.upper()}\nSharpe: {result["sharpe"]:.3f}', fontweight='bold')

plt.suptitle('🥧 Optimal Portfolio Allocations', fontsize=16, y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Generate efficient frontier
print("📊 Generating efficient frontier...")

try:
    target_returns, frontier_vols, frontier_sharpes = optimizer.efficient_frontier(
        next_returns, cov_matrix, num_points=30
    )
    
    plt.figure(figsize=(12, 8))
    
    # Plot efficient frontier
    plt.plot(frontier_vols, target_returns, 'b-', linewidth=3, label='Efficient Frontier')
    
    # Plot optimization results
    for method, result in optimization_results.items():
        plt.scatter(result['volatility'], result['expected_return'], 
                   s=150, label=f'{method.upper()}', alpha=0.8)
    
    # Plot individual assets
    asset_vols = np.sqrt(np.diag(cov_matrix))
    plt.scatter(asset_vols, next_returns, s=100, alpha=0.6, color='red', marker='x')
    
    # Add asset labels
    for i, ticker in enumerate(tickers):
        plt.annotate(ticker, (asset_vols[i], next_returns.iloc[i]), 
                    xytext=(5, 5), textcoords='offset points')
    
    plt.xlabel('Volatility (Risk)')
    plt.ylabel('Expected Return')
    plt.title('📊 Efficient Frontier and Portfolio Optimization Results', fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
except Exception as e:
    print(f"❌ Efficient frontier generation failed: {e}")

## 6. 🔄 Backtesting & Performance Evaluation

Finally, let's backtest our strategy and evaluate its performance.

In [ ]:
# For demonstration, let's create a simple strategy using Sharpe-optimal weights
print("🔄 Setting up backtesting...")

# Use Sharpe-optimal weights as our strategy
if 'sharpe' in optimization_results:
    optimal_weights = optimization_results['sharpe']['weights']
else:
    # Fallback to equal weights
    optimal_weights = np.ones(len(tickers)) / len(tickers)

# Create weights DataFrame (constant weights for simplicity)
weights_df = pd.DataFrame(
    index=price_data.index,
    columns=tickers,
    data=np.tile(optimal_weights, (len(price_data), 1))
)

print(f"📊 Strategy weights: {dict(zip(tickers, optimal_weights))}")

In [ ]:
# Initialize backtester
config = TradingConfig()
config.risk_free_rate = 0.02

backtester = Backtester(
    config=config,
    initial_capital=100000,
    transaction_cost=0.001,
    rebalance_frequency='monthly',
    benchmark_ticker='SPY'
)

# Run backtest
print("🚀 Running backtest...")

benchmark_data = price_data[['SPY']]  # Use SPY as benchmark

backtest_results = backtester.run_backtest(
    price_data=price_data,
    weight_data=weights_df,
    benchmark_data=benchmark_data
)

print("✅ Backtest completed!")

In [ ]:
# Display results summary
summary = backtest_results.summary()

print("📊 BACKTEST RESULTS SUMMARY")
print("=" * 50)

for metric, value in summary.items():
    if isinstance(value, float):
        if 'Return' in metric or 'Alpha' in metric:
            print(f"{metric:.<30} {value:.2%}")
        elif 'Ratio' in metric or 'Beta' in metric:
            print(f"{metric:.<30} {value:.3f}")
        elif 'Costs' in metric:
            print(f"{metric:.<30} ${value:,.2f}")
        else:
            print(f"{metric:.<30} {value:.4f}")
    else:
        print(f"{metric:.<30} {value}")

In [ ]:
# Plot backtest results
backtester.plot_results(figsize=(15, 10))

In [ ]:
# Performance evaluation
evaluator = PerformanceEvaluator(risk_free_rate=0.02)

# Generate comprehensive report
performance_report = evaluator.generate_report(
    returns=backtest_results.portfolio_returns,
    benchmark_returns=backtest_results.benchmark_returns,
    strategy_name="Demo Strategy"
)

print(performance_report)

## 📝 Summary & Insights

This notebook demonstrated the complete algorithmic trading pipeline:

### ✅ **What We Accomplished**

1. **📊 Data Pipeline**: Successfully loaded and processed multi-asset financial data
2. **🔧 Feature Engineering**: Computed comprehensive technical indicators (RSI, MACD, MA, etc.)
3. **🔮 Forecasting**: Generated return and volatility forecasts using ARIMA+GARCH models
4. **📡 Signal Generation**: Created and combined multiple trading strategies
5. **📈 Portfolio Optimization**: Optimized portfolio weights using modern portfolio theory
6. **🔄 Backtesting**: Simulated realistic trading with transaction costs and benchmarking
7. **📊 Performance Evaluation**: Comprehensive risk and return analysis

### 🎯 **Key Insights**

- **Diversification Benefits**: Multi-asset portfolios show improved risk-adjusted returns
- **Signal Combination**: Ensemble methods provide more robust trading signals
- **Transaction Costs Matter**: Realistic cost modeling significantly impacts strategy performance
- **Risk Management**: Volatility forecasting enables better risk-adjusted position sizing

### 🚀 **Next Steps**

- Experiment with different rebalancing frequencies
- Add more sophisticated risk models
- Implement online learning for adaptive strategies
- Explore reinforcement learning agents
- Add alternative data sources

---

**⚠️ Disclaimer**: This is for educational purposes only. Past performance does not guarantee future results. Always consult financial professionals before making investment decisions.